# 01 - Loading and joining the data

In this notebook I load the two Kaggle train files, join them into one table and check the basic numbers: rows, columns, fraud rate and time range.

## Why I do EDA

The model needs **clues**: columns where fraud looks different from genuine payments. EDA (exploratory data analysis) is how I find out, before training, which columns are good clues, which are useless, and which need fixing.

For every check I ask: **will this help the model catch fraud, and how should I prepare it?**

| Check | Why | Used later for |
|---|---|---|
| Fraud rate | how rare fraud is | class weights, PR-AUC instead of accuracy |
| Time range | how to split | time-based train / validation / test |
| Duplicates | repeated rows distort learning | data is clean |
| Missing values | some models cannot handle gaps | which columns to drop or fill |
| Fraud with / without device data | is "missing" itself a clue? | "was missing" flag |
| Amount | is it a clue, is it skewed? | log transform for Logistic Regression |
| Fraud rate by category | which groups are riskiest? | which features to keep |

## Data notes (from the Kaggle data page)

The data is broken into two files `identity` and `transaction`, which are joined by `TransactionID`. Not all transactions have corresponding identity information.
Categorical Features - Transaction

* `ProductCD`
* `card1` - `card6`
* `addr1`, `addr2`
* `P_emaildomain`
* `R_emaildomain`
* `M1` - `M9`

Categorical Features - Identity

* `DeviceType`
* `DeviceInfo`
* `id_12` - `id_38`

The `TransactionDT` feature is a timedelta from a given reference datetime (not an actual timestamp).
You can read more about the data from [this post by the competition host](https://www.kaggle.com/c/ieee-fraud-detection/discussion/101203).
Files

* train_{transaction, identity}.csv - the training set
* test_{transaction, identity}.csv - the test set (you must predict the `isFraud` value for these observations)
* sample_submission.csv - a sample submission file in the correct format

## 1. Transaction file

This is the main file. Each row is one payment: amount, card, product, email domain and `isFraud`, which is the answer the model learns from.

In [19]:
# pandas for loading and working with the tables
import pandas as pd

# path goes up one folder from notebooks/ to reach data/raw/
df = pd.read_csv("../data/raw/train_transaction.csv")
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
# (rows, columns) of the transaction table
print(df.shape)

(590540, 394)


**Result:** 590,540 transactions and 394 columns.

## 2. Identity file

This file has device information for each payment: device type, operating system, browser, screen size. I use the **train** file because only the train data has `isFraud` labels.

In [21]:
import pandas as pd

# train identity file, not test: test has no isFraud labels and its IDs would not match
df1 = pd.read_csv("../data/raw/train_identity.csv")
df1.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


In [22]:
# (rows, columns) of the identity table
df1.shape

(144233, 41)

**Result:** 144,233 rows and 41 columns. So only about 24% of transactions have device data.

## 3. Joining the two tables

The model needs one table where each row is one transaction with everything known about it.

- **Merge, not concat:** concat lines rows up by position, but the two files are not in the same order (identity row 0 is transaction 2987004, which is row 4 in the transaction file). Merge matches rows by `TransactionID`.
- **Left join, not inner:** an inner join keeps only transactions that have device data and would delete about 76% of the data. A left join keeps every transaction, and the device columns are empty (NaN) where there is no identity row. In real life the model has to score payments without device data too.

In [23]:
# left join on TransactionID: keeps every transaction, device columns are NaN where there is no identity row
df_merged = pd.merge(df, df1, on='TransactionID', how='left')
df_merged.shape

(590540, 434)

**Result:** 590,540 rows (no transaction lost) and 434 columns (394 + 41, minus 1 because `TransactionID` appears only once).

In [24]:
# row labels pandas adds (0, 1, 2 ...), not the transaction ID
df_merged.index


RangeIndex(start=0, stop=590540, step=1)

In [25]:
# quick look at the merged table
df_merged


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Fraud rate

`isFraud` is 0 for a genuine payment and 1 for fraud. Dividing the count of each value by the total number of rows gives the share of each.

In [26]:
# share of genuine (0) and fraud (1): count of each value divided by the total number of rows
df_merged['isFraud'].value_counts()/len(df_merged)

isFraud
0    0.96501
1    0.03499
Name: count, dtype: float64

**Result:** 569,877 genuine (96.5%) and 20,663 fraud (3.5%). Fraud is about 1 in every 29 transactions, so the data is heavily imbalanced.

**Why accuracy is not enough:** a model that says "not fraud" for every transaction would get 96.5% accuracy and still catch zero fraud. So I will measure recall (how much of the real fraud is caught), precision (how many alerts are really fraud) and PR-AUC, and use class weights so the model does not ignore the rare fraud cases.

## 5. Time range

`TransactionDT` is not a real date. It is the number of seconds since a hidden reference point. Dividing by 86,400 (seconds in a day) turns it into days.

In [27]:
# TransactionDT is in seconds from a hidden start point; 86400 seconds = 1 day
print(df_merged['TransactionDT'].min()/86400)
print(df_merged['TransactionDT'].max()/86400)

1.0
182.99920138888888


**Result:** the data starts on day 1 and ends on day 183, so it covers about 182 days, roughly 6 months.

**Why this matters for the split:** in real life the model is trained on past payments and has to catch fraud in future ones. So I will split by time: the earliest months for training, the next part for validation and the last part for testing.

A random split would be cheating. Fraudsters often make many payments with the same card in a short time. With a random split, some of those payments end up in training and the rest in testing, so the model can simply remember the card instead of learning what fraud looks like. The score looks great but would not hold up on new fraudsters next month. A time split gives a lower but honest score.

## 6. Duplicates

Each payment should appear only once. If a transaction were repeated, the model would learn from it twice and the fraud counts would be wrong.

In [28]:
# True if this TransactionID was already seen above; counting True/False shows how many duplicates there are
print(df_merged['TransactionID'].duplicated().sum())

0


**Result:** all 590,540 values are False, so there are no duplicate transactions. Every `TransactionID` is unique, and the left join did not create extra rows.

## Day 1 summary

| Check | Result |
|---|---|
| Transaction file | 590,540 rows, 394 columns |
| Identity file | 144,233 rows, 41 columns (about 24% of transactions have device data) |
| Merged (left join) | 590,540 rows, 434 columns |
| Fraud rate | 3.5% (20,663 fraud) |
| Time range | day 1 to day 183, about 6 months |
| Duplicates | 0 |

**Next:** missing values and data quality.

# Day 2 - Data quality

## 7. Missing values

Many columns have empty values (NaN). I check how much is missing in each column, first as a count and then as a percentage.

In [29]:
# number of empty (NaN) values in each column
df_merged.isnull().sum()

TransactionID          0
isFraud                0
TransactionDT          0
TransactionAmt         0
ProductCD              0
                   ...  
id_36             449555
id_37             449555
id_38             449555
DeviceType        449730
DeviceInfo        471874
Length: 434, dtype: int64

**Result:** the key columns (`TransactionID`, `isFraud`, `TransactionDT`, `TransactionAmt`, `ProductCD`) have no missing values. `DeviceType` and `DeviceInfo` are missing for about 450,000-470,000 rows, which matches the transactions with no identity row after the left join.

In [30]:
# share of empty values in each column, as a percentage (average of True/False x 100)
df_merged.isnull().mean()*100

TransactionID      0.000000
isFraud            0.000000
TransactionDT      0.000000
TransactionAmt     0.000000
ProductCD          0.000000
                    ...    
id_36             76.126088
id_37             76.126088
id_38             76.126088
DeviceType        76.155722
DeviceInfo        79.905510
Length: 434, dtype: float64

**Result:** `id_36` to `id_38` are 76.1% empty, exactly the share of transactions with no identity data (100% - 24%). `DeviceInfo` is 79.9% empty, so some transactions have an identity row but still no device name.

- First line: stores the missing percentage for every column.
- Second line: shows the 10 columns with the highest missing percentages.
ascending=False means largest to smallest.

In [31]:
# missing % per column, saved so I can sort and filter it
missing_pct = df_merged.isnull().mean() * 100

# 10 emptiest columns first
missing_pct.sort_values(ascending=False).head(10)

id_24    99.196159
id_25    99.130965
id_07    99.127070
id_08    99.127070
id_21    99.126393
id_26    99.125715
id_27    99.124699
id_23    99.124699
id_22    99.124699
dist2    93.628374
dtype: float64

**Result:** the emptiest columns are identity columns like `id_24`, `id_25`, `id_07` and `id_08`, which are about 99% empty. `dist2` is the emptiest transaction column at 93.6%. Before dropping any of them I will check whether "missing" itself is linked to fraud.

### How many columns are almost empty, and how many are complete?

I filter `missing_pct` with a condition in square brackets and count what is left with `len`.

In [32]:
# Columns with more than 90% missing
print(len(missing_pct[missing_pct > 90]))

# Columns with no missing values
print(len(missing_pct[missing_pct == 0]))

12
20


**Result:** 12 columns are more than 90% empty, and only 20 of the 434 columns have no missing values. The other 414 columns all have gaps, which is normal for real payment data. Handling missing values will be a big part of preprocessing.

**Next question:** is a missing value itself a clue? I will compare the fraud rate for transactions with and without device data.

## 8. Is missing device data a clue?

I split the transactions into two groups, with device data and without it (based on whether `DeviceType` is empty), and compare their fraud rates.

A mask is a column of True/False values. Putting it inside square brackets keeps only the rows where it is True.

In [33]:
# Identify rows where DeviceType is present
device_mask = df_merged['DeviceType'].notna()

# Keep those transaction rows
with_device = df_merged[device_mask]

# Calculate their fraud percentage
print(with_device['isFraud'].mean() * 100)

7.961792486329096


**Result:** transactions **with** device data have a fraud rate of about 7.96%, more than double the overall 3.5%. So whether device data is present is itself linked to fraud, and I should not simply throw that information away.

In [34]:
# Identify rows where DeviceType is missing
missing_device_mask = df_merged['DeviceType'].isna()

# Keep those transaction rows
without_device = df_merged[missing_device_mask]

# Calculate their fraud percentage
print(without_device['isFraud'].mean() * 100)

2.1017054677250795


**Result:** transactions **without** device data have a fraud rate of about 2.10%.

| Group | Share of transactions | Fraud rate |
|---|---|---|
| With device data | 24% | 7.96% |
| Without device data | 76% | 2.10% |
| All | 100% | 3.5% |

The overall 3.5% is a mix of the two groups, so if one is above it the other has to be below. Transactions with device data are about 4 times more likely to be fraud (7.96 / 2.10).

**Finding 1:** whether device data is missing is a fraud signal on its own. When I fill missing values for Logistic Regression, I will add a 0/1 "was missing" flag so the model keeps this information. XGBoost can handle missing values directly.

## 9. Payment amount (`TransactionAmt`)

How big are the payments? I look at the smallest, largest, mean and median amount.

- **Mean:** add up all amounts and divide by the number of payments. Extreme values pull it up or down.
- **Median:** the middle payment when they are sorted. Half are smaller, half are bigger. Extreme values barely move it, so it is called **robust**.

In [35]:
# smallest, largest, average (mean) and middle (median) payment amount
print(df_merged['TransactionAmt'].min())
print(df_merged['TransactionAmt'].max())
print(df_merged['TransactionAmt'].mean())
print(df_merged['TransactionAmt'].median())

0.251
31937.391
135.02717637247264
68.769


**Result:**

| | Amount |
|---|---|
| Smallest | $0.25 |
| Median | $68.77 |
| Mean | $135.03 |
| Largest | $31,937.39 |

The mean is about twice the median. Most payments are small (half are under $69), but a few very large ones pull the average up. So the amounts are **right-skewed**.

The smallest payment is only $0.25. Tiny payments can be a fraud pattern: fraudsters sometimes test a stolen card with a small amount before making big purchases.

### Skewness: right, left and symmetric

| Shape | Mean vs median | Long tail on the... | Example |
|---|---|---|---|
| **Right skew** (positive) | mean **>** median | right: a few very **big** values | incomes, house prices, these payment amounts |
| **Left skew** (negative) | mean **<** median | left: a few very **small** values | marks on an easy exam: most score high, a few score very low |
| **Symmetric** | mean **≈** median | both sides balanced | people's heights |

The skew is named after where the tail points. The mean gets pulled toward the tail because every value counts in the average. The median only looks at the middle, so it stays put.

**Example:** payments of $10, $20, $30, $40 and $900. Mean = $200, median = $30. One big payment moved the mean a lot and the median not at all.

**Why it matters for this project:**
- **Logistic Regression** is sensitive to extreme values, so I may use the log of the amount, which pulls big values closer together.
- **XGBoost** splits on thresholds, so skew does not bother it.
- **Business cost:** a few big frauds can cost more than many small ones, which is why I also measure fraud money caught.

### Fraud vs genuine amounts

**Question:** are fraud payments bigger or smaller than genuine ones? If they clearly differ, the amount is a useful clue for the model.

`groupby` splits the table into groups by the values of one column (here `isFraud`), then calculates something for each group. I check both the mean and the median, because the mean can be pulled up by a few huge payments.

In [38]:
# split the table into genuine (0) and fraud (1), then take the mean / median amount of each group
# (a notebook only shows the last line automatically; wrap in print() to see both)
df_merged.groupby('isFraud')['TransactionAmt'].mean()
df_merged.groupby('isFraud')['TransactionAmt'].median()

isFraud
0    68.5
1    75.0
Name: TransactionAmt, dtype: float64

**Result:**

| | Mean | Median |
|---|---|---|
| Genuine (0) | $134.51 | $68.50 |
| Fraud (1) | $149.24 | $75.00 |

Fraud payments are a bit bigger by **both** measures. Because the median is higher too, it is not just a few big frauds pulling the average up: the typical fraud really is slightly larger. But the gap is small (about $6.50 on the median).

**Finding 2:** fraud payments are slightly larger than genuine ones (median $75 vs $68.50), but the difference is small, so the amount alone is a weak signal. The model will have to combine it with other clues such as device, product and card.

## 10. Fraud rate by category

For each category column I ask: is fraud much more common in some groups than others? If yes, the column is a strong clue for the model. If every group is close to 3.5%, it does not help much.

### Product type (`ProductCD`)

`ProductCD` is the type of product bought (codes C, H, R, S, W). I group by it and take the mean of `isFraud`, which gives the fraud rate of each product type.

*Note:* the column I split by goes inside `groupby(...)`, and the column I calculate on goes in the square brackets. The other way round fails, because `ProductCD` is text and text cannot be averaged.

In [42]:
# fraud rate per product type: group by ProductCD, then average isFraud (mean of 0/1 = share of fraud)
print(df_merged.groupby('ProductCD')['isFraud'].mean())


ProductCD
C    0.116873
H    0.047662
R    0.037826
S    0.058996
W    0.020399
Name: isFraud, dtype: float64


**Result:**

| ProductCD | Fraud rate |
|---|---|
| C | 11.7% |
| S | 5.9% |
| H | 4.8% |
| R | 3.8% |
| W | 2.0% |
| *Overall* | *3.5%* |

Product type **C** has more than three times the average fraud rate, and about six times the rate of **W**. That is a big difference, so `ProductCD` looks like a strong clue, much stronger than the amount.

**Still to check:** how many transactions each product type has. A high rate from a very small group could just be luck.

### How big is each product group?

A high fraud rate only means something if the group is big enough. So I count the genuine and fraud transactions in each product type. The output has two levels of labels on the left (product, then 0/1), which pandas calls a **MultiIndex**.

In [44]:
# genuine (0) and fraud (1) counts inside each product type
print(df_merged.groupby('ProductCD')['isFraud'].value_counts())

ProductCD  isFraud
C          0           60511
           1            8008
H          0           31450
           1            1574
R          0           36273
           1            1426
S          0           10942
           1             686
W          0          430701
           1            8969
Name: count, dtype: int64


**Result:**

| ProductCD | Transactions | Share | Genuine | Fraud | Fraud rate |
|---|---|---|---|---|---|
| W | 439,670 | 74% | 430,701 | 8,969 | 2.0% |
| C | 68,519 | 12% | 60,511 | 8,008 | 11.7% |
| R | 37,699 | 6% | 36,273 | 1,426 | 3.8% |
| H | 33,024 | 6% | 31,450 | 1,574 | 4.8% |
| S | 11,628 | 2% | 10,942 | 686 | 5.9% |

**Check:** 8,008 + 1,574 + 1,426 + 686 + 8,969 = 20,663, the same total fraud count as on Day 1.

Product C is a big group (68,519 transactions), so its high rate is real and not luck. C has almost as many fraud cases as W (8,008 vs 8,969), even though W has six times more transactions.

**Finding 3:** product type C is the riskiest (11.7% fraud, more than three times the average). It is only 12% of transactions but holds about 39% of all fraud (8,008 of 20,663). W is the most common (74%) and the safest (2.0%). A fraud team would send more product C payments for review.

### Card network (`card4`) and card type (`card6`)

Does the card change the fraud risk? `card4` is the card network (visa, mastercard, discover, american express) and `card6` is the card type (debit, credit). I count genuine and fraud transactions in each group and work out the fraud rate as fraud / (genuine + fraud).

In [45]:
# genuine (0) and fraud (1) counts for each card network (card4) and card type (card6)
print(df_merged.groupby('card4')['isFraud'].value_counts())
print(df_merged.groupby('card6')['isFraud'].value_counts())

card4             isFraud
american express  0            8089
                  1             239
discover          0            6137
                  1             514
mastercard        0          182721
                  1            6496
visa              0          371394
                  1           13373
Name: count, dtype: int64
card6            isFraud
charge card      0              15
credit           0          139036
                 1            9950
debit            0          429264
                 1           10674
debit or credit  0              30
Name: count, dtype: int64


**Result: card network (`card4`)**

| card4 | Transactions | Fraud | Fraud rate |
|---|---|---|---|
| visa | 384,767 | 13,373 | 3.5% |
| mastercard | 189,217 | 6,496 | 3.4% |
| american express | 8,328 | 239 | 2.9% |
| discover | 6,651 | 514 | 7.7% |

Visa and mastercard are almost exactly at the average (3.5%), so for most cards the network says little. Discover is more than twice as risky, but it is a small group (about 1% of transactions).

**Result: card type (`card6`)**

| card6 | Transactions | Fraud | Fraud rate |
|---|---|---|---|
| debit | 439,938 | 10,674 | 2.4% |
| credit | 148,986 | 9,950 | 6.7% |
| debit or credit | 30 | 0 | 0% |
| charge card | 15 | 0 | 0% |

Credit cards have about 2.7 times the fraud rate of debit cards. Credit is only a quarter of transactions but has almost half of all fraud (9,950 of 20,663). "debit or credit" and "charge card" are too small (45 rows together) to say anything about.

**Note:** the fraud counts add up to 20,622 for `card4` and 20,624 for `card6`, not 20,663. `groupby` leaves out rows where the column is empty (NaN), so about 40 fraud cases have no card network or type.

**Finding 4:** credit cards are much riskier than debit cards (6.7% vs 2.4% fraud). The card network matters less: visa and mastercard sit at the average, and only the small discover group stands out (7.7%).

In [ ]:
print(df_merged.groupby('P_emaildomain')['isFraud'].value_counts())

P_emaildomain  isFraud
aim.com        0            275
               1             40
anonymous.com  0          36139
               1            859
aol.com        0          27672
                          ...  
yahoo.es       1              2
yahoo.fr       0            138
               1              5
ymail.com      0           2346
               1             50
Name: count, Length: 101, dtype: int64


### Email domain (`P_emaildomain`)

`P_emaildomain` is the email provider of the buyer (gmail.com, yahoo.com...). There are 59 different domains, and many are tiny. A domain with 2 transactions and 1 fraud would show 50% fraud just by luck, so I only trust rates from big groups.

`agg` runs several calculations in one go. With `['mean', 'count']` I get the fraud rate and the number of transactions for every domain in one table. Then I sort by `count` and look at the 10 most common domains.

In [ ]:
# fraud rate (mean of isFraud) and number of transactions (count) for every email domain
email_stats = df_merged.groupby('P_emaildomain')['isFraud'].agg(['mean', 'count'])

# sorting a table needs 'by': sort by the count column, biggest first, and keep the top 10
email_stats.sort_values(by='count', ascending=False).head(10)

**Result: 10 most common email domains**

| P_emaildomain | Transactions | Fraud rate |
|---|---|---|
| gmail.com | 228,355 | 4.4% |
| yahoo.com | 100,934 | 2.3% |
| hotmail.com | 45,250 | 5.3% |
| anonymous.com | 36,998 | 2.3% |
| aol.com | 28,289 | 2.2% |
| comcast.net | 7,888 | 3.1% |
| icloud.com | 6,267 | 3.1% |
| outlook.com | 5,096 | 9.5% |
| msn.com | 4,092 | 2.2% |
| att.net | 4,033 | 0.7% |

**outlook.com** stands out at 9.5%, almost three times the average, from over 5,000 transactions. **hotmail.com** (5.3%) and **gmail.com** (4.4%) are also above average, and gmail is so big that it holds a large share of all fraud. **yahoo.com**, **aol.com** and **msn.com** are low (about 2.2%), and **att.net** is very low (0.7%).

About 16% of transactions have no email domain at all, and their fraud rate is about 3.0%, close to the average.

**Finding 5:** the email provider is a useful clue. outlook.com (9.5%) and hotmail.com (5.3%) are riskier, while yahoo.com, aol.com and att.net are safer. Small domains like aim.com (12.7% from only 315 transactions) look scary but are too small to trust.

## 11. Fraud over time

**Why:** I will split the data by time (train on early weeks, test on later ones). If fraud is very different in early and late weeks, the model learns from a world that has changed by test time. This check shows whether fraud is steady or shifting.

First I turn `TransactionDT` (seconds) into a week number. One week is 86,400 x 7 = 604,800 seconds. `//` divides and drops the decimals, so every transaction gets a whole week number from 0 to 26. I use weeks rather than days because daily rates jump around a lot by chance.

In [51]:
# new column: week number of each transaction (seconds // seconds in a week; // drops the decimals)
df_merged['week'] = df_merged['TransactionDT'] // (86400 * 7)
print(df_merged['week'])

0          0
1          0
2          0
3          0
4          0
          ..
590535    26
590536    26
590537    26
590538    26
590539    26
Name: week, Length: 590540, dtype: int64


**Result:** weeks run from 0 to 26, and the rows are already in time order (week 0 at the top, week 26 at the bottom). That makes the time split easy.

In [53]:
# fraud rate for each week (mean of isFraud inside each week)
df_merged.groupby('week')['isFraud'].mean()

week
0     0.028055
1     0.026148
2     0.025500
3     0.018523
4     0.038226
5     0.038321
6     0.038594
7     0.041199
8     0.045540
9     0.042973
10    0.043145
11    0.034542
12    0.039017
13    0.033669
14    0.032647
15    0.041498
16    0.050612
17    0.036770
18    0.035318
19    0.034568
20    0.035915
21    0.031652
22    0.028496
23    0.032515
24    0.036672
25    0.041627
26    0.043210
Name: isFraud, dtype: float64

**Result:**

| Weeks | Fraud rate | Note |
|---|---|---|
| 0-3 | 1.9% - 2.8% | lowest, and weeks 2-3 are the busiest (34,000-37,000 transactions) |
| 4-10 | 3.8% - 4.6% | above average |
| 11-25 | 2.9% - 5.1% | around the average, peak 5.1% in week 16 |
| 26 | 4.3% | only 2,754 transactions, a partial week |

Fraud is not perfectly steady. The first four weeks have clearly lower fraud and more transactions (possibly a busy shopping season), and after that the rate moves between about 3% and 5%.

**What this means for the split:**
- The fraud rate in train, validation and test will not be exactly the same, so I will check it in each part.
- The last weeks (about 3-4%) are close to the overall 3.5%, so the test period is not unusual.
- Week 26 is incomplete, so I will not read much into it on its own.

### Fraud rate per week as a chart

The same weekly numbers as a line, with a dashed line at the overall 3.5% so it is easy to see which weeks are above or below average. I also save the chart as an image for the README.

In [ ]:
import matplotlib.pyplot as plt

# fraud rate per week, as a percentage
weekly_fraud = df_merged.groupby('week')['isFraud'].mean() * 100

# line chart: one point per week
ax = weekly_fraud.plot(figsize=(9, 4), marker='o', label='weekly fraud rate')

# dashed line at the overall fraud rate, to compare each week against it
ax.axhline(df_merged['isFraud'].mean() * 100, color='gray', linestyle='--', label='overall 3.5%')

# title and axis labels so the chart explains itself
ax.set_title('Fraud rate per week')
ax.set_xlabel('Week')
ax.set_ylabel('Fraud rate (%)')
ax.legend()
plt.tight_layout()

# save a copy for the README
plt.savefig('../reports/figures/weekly_fraud_rate.png', dpi=120)
plt.show()

**What the chart shows:** a clear dip in weeks 0-3 (lowest 1.9% in week 3), a jump to about 4% from week 4, a peak of 5.1% in week 16, and a dip again around week 22 (2.9%) before rising to about 4% at the end. Most weeks stay between 3% and 4.5%.